# Coupled cluster theory

Companion notebook to Chapter 10 of *Quantum mechanics for many-particle
systems*.  Every number quoted in the chapter is produced here; the code is
the same as in `BookManybody/BookMaterial/Programs/coupledcluster.py`.

The ansatz is a single change from configuration interaction:

$$|\Psi\rangle = e^{\hat T}|\Phi_0\rangle,
\qquad \hat T = \hat T_1 + \hat T_2 + \cdots$$

Requiring $\overline H = e^{-\hat T}\hat H e^{\hat T}$ to have the reference
as an eigenstate gives

$$\Delta E = \langle\Phi_0|\overline H_N|\Phi_0\rangle,\qquad
  0 = \langle\Phi_i^a|\overline H_N|\Phi_0\rangle,\qquad
  0 = \langle\Phi_{ij}^{ab}|\overline H_N|\Phi_0\rangle.$$

Contents:

1. Validating the solver
2. The pairing model
3. What coupled cluster contains, order by order
4. When singles matter
5. Size extensivity
6. Everything against everything

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join("..", "BookManybody", "BookMaterial", "Programs"))
import coupledcluster as cc

np.set_printoptions(precision=6, suppress=True, linewidth=120)

## 1. Validating the solver

Four checks: the models must reproduce the exact energies of Chapters 4 and
7; the singles must vanish for the pairing model; the first iteration must be
MP2; and the Hartree-Fock rotation must leave the exact energy alone while
making $f_{ia}$ vanish.

In [ ]:
cc.demo_validation()

## 2. The pairing model

Four doubly degenerate levels, four particles.  The reference energy is the
Hartree-Fock energy $2-g$ of Chapter 6, so everything below is correlation
energy.

In [ ]:
cc.demo_pairing()

In [ ]:
gs = np.linspace(0.1, 2.2, 22)
ref, mp2, ccd_e, exact = [], [], [], []
for g in gs:
    h, v, N = cc.pairing_model(g=g)
    fk = cc.fock_matrix(h, v, N)
    e0 = cc.reference_energy(h, v, N)
    ref.append(e0)
    mp2.append(e0 + cc.mp2_energy(fk, v, N))
    ccd_e.append(e0 + cc.ccd(fk, v, N)["energy"])
    exact.append(cc.fci_energy(h, v, N)[0])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
ax[0].plot(gs, exact, "k-o", ms=3, label="exact (FCI)")
ax[0].plot(gs, ref, "C7--", label="Hartree-Fock")
ax[0].plot(gs, mp2, "C0-", label="MP2")
ax[0].plot(gs, ccd_e, "C3-", label="CCD")
ax[0].set_xlabel("$g$"); ax[0].set_ylabel("ground-state energy")
ax[0].legend()
ax[1].semilogy(gs, np.abs(np.array(mp2) - np.array(exact)), "C0-", label="MP2")
ax[1].semilogy(gs, np.abs(np.array(ccd_e) - np.array(exact)), "C3-",
               label="CCD")
ax[1].set_xlabel("$g$"); ax[1].set_ylabel("|error|")
ax[1].set_title("CCD is exact to four parts in a million at weak coupling")
ax[1].legend()
fig.tight_layout(); plt.show()

## 3. What coupled cluster contains, order by order

The CCD equation is exactly quadratic in the amplitudes, so it splits as
$R(t) = C + L[t] + Q[t,t]$ and can be expanded in powers of the interaction.
The result is Møller-Plesset perturbation theory — verified here against the
independent implementation of Chapter 9.

In [ ]:
cc.demo_orders()

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for g, style in ((0.5, "-o"), (1.0, "-s"), (1.5, "-^")):
    h, v, N = cc.pairing_model(g=g)
    fk = cc.fock_matrix(h, v, N)
    e, _ = cc.ccd_order_by_order(fk, v, N, order=16)
    ax.semilogy(range(2, 18), np.abs(e), style, ms=4, label=f"$g = {g}$")
ax.set_xlabel("order in the interaction")
ax.set_ylabel(r"$|\Delta E^{(n)}|$")
ax.set_title("the perturbative content of CCD")
ax.legend(); fig.tight_layout(); plt.show()

The expansion converges back onto the CCD energy.  Solving the nonlinear
equation reaches the same answer in twenty-odd iterations without ever
referring to the order structure — that is what "infinite resummation" means
in practice.

## 4. When singles matter

The particle-hole term of Chapter 7 breaks pairs, so the reference couples to
$1p$-$1h$ states and the singles amplitudes are no longer zero.  We run CCSD
twice: on the bare oscillator reference and on the Hartree-Fock reference,
where Brillouin's theorem makes the singles start one order later.

In [ ]:
cc.demo_ccsd()

In [ ]:
fs = np.linspace(0.0, 0.5, 11)
g = 0.5
d_e, s_e, ex_e, t1max = [], [], [], []
for f in fs:
    h, v, N = cc.pairing_ph_model(g=g, f=f)
    fk = cc.fock_matrix(h, v, N)
    e0 = cc.reference_energy(h, v, N)
    d_e.append(e0 + cc.ccd(fk, v, N, mixing=0.3)["energy"])
    s = cc.ccsd(fk, v, N, mixing=0.3)
    s_e.append(e0 + s["energy"]); t1max.append(np.abs(s["t1"]).max())
    ex_e.append(cc.fci_energy(h, v, N)[0])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(fs, ex_e, "k-o", ms=3, label="exact")
ax[0].plot(fs, d_e, "C0-", label="CCD")
ax[0].plot(fs, s_e, "C3-", label="CCSD")
ax[0].set_xlabel("$f$"); ax[0].set_ylabel("ground-state energy")
ax[0].set_title(f"$g = {g}$"); ax[0].legend()
ax[1].plot(fs, t1max, "C2-o", ms=3)
ax[1].set_xlabel("$f$"); ax[1].set_ylabel(r"$\max|t_1|$")
ax[1].set_title("the singles grow with the pair-breaking term")
fig.tight_layout(); plt.show()

## 5. Size extensivity

For non-interacting subsystems the cluster operators commute, so
$e^{\hat T_A + \hat T_B} = e^{\hat T_A}e^{\hat T_B}$, the wave function
factorises and the energy is additive — at any truncation level.  This is the
property truncated configuration interaction cannot have.

In [ ]:
cc.demo_extensivity()

## 6. Everything against everything

In [ ]:
cc.demo_comparison()

## 7. Unitary coupled cluster

Replacing $e^{\hat T}$ by $e^{\hat T - \hat T^\dagger}$ makes the exponential
*unitary*, so the state stays normalised and the energy

$$E(\mathbf t) = \langle\Phi_0|e^{-\hat\sigma}\hat H e^{\hat\sigma}|\Phi_0\rangle,
\qquad \hat\sigma = \hat T - \hat T^\dagger$$

is a genuine expectation value — **variational**, bounded below by $E_0$.

The price is that $[\hat T, \hat T^\dagger]\neq 0$, so the
Baker-Campbell-Hausdorff series no longer terminates and the exponential must
be built explicitly.  On a classical computer that is expensive; on a quantum
computer the exponential is *applied* rather than expanded, and the bound is
exactly what an optimiser wants.

In [ ]:
cc.demo_unitary()

In [ ]:
gs = np.linspace(0.1, 2.2, 15)
e_ccd, e_uccd, e_exact = [], [], []
for g in gs:
    h, v, N = cc.pairing_model(g=g)
    fk = cc.fock_matrix(h, v, N)
    e0 = cc.reference_energy(h, v, N)
    e_ccd.append(e0 + cc.ccd(fk, v, N)["energy"])
    u = cc.UnitaryCC(h, v, N, pair_only=True)
    e_uccd.append(u.optimise(restarts=2)["energy"])
    e_exact.append(u.exact())
e_ccd, e_uccd, e_exact = map(np.array, (e_ccd, e_uccd, e_exact))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
ax[0].plot(gs, e_exact, "k-o", ms=3, label="exact (FCI)")
ax[0].plot(gs, e_ccd, "C0-", label="CCD")
ax[0].plot(gs, e_uccd, "C3-", label="UCCD")
ax[0].set_xlabel("$g$"); ax[0].set_ylabel("ground-state energy"); ax[0].legend()
ax[1].plot(gs, e_ccd - e_exact, "C0-o", ms=3, label="CCD")
ax[1].plot(gs, e_uccd - e_exact, "C3-s", ms=3, label="UCCD")
ax[1].axhline(0.0, color="k", lw=0.8)
ax[1].set_xlabel("$g$"); ax[1].set_ylabel("signed error")
ax[1].set_title("UCCD stays above, CCD below")
ax[1].legend()
fig.tight_layout(); plt.show()

The signed-error panel is the point: every UCCD error is positive, as the
variational principle demands, and every CCD error is negative.  The two are
the *same truncation of the same cluster operator* — only the unitarity of the
exponential differs.

## 8. Trotterisation

A quantum circuit cannot apply $e^{\hat\sigma}$ in one step.  It applies each
generator separately,

$$e^{\hat\sigma(\mathbf t)} \approx
  \Big[\prod_k e^{t_k(\hat A_k - \hat A_k^\dagger)/n}\Big]^n,$$

which is the Trotter splitting of Chapter 6 with the cluster generators in
place of the pieces of a Hamiltonian.

In [ ]:
cc.demo_trotter()

In [ ]:
h, v, N = cc.pairing_model(g=1.0)
u = cc.UnitaryCC(h, v, N, pair_only=True)
ref = u.optimise(restarts=2)
t_star = ref["amplitudes"]
psi = u.state(t_star); psi /= np.linalg.norm(psi)

ns = np.array([1, 2, 4, 8, 16, 32, 64])
state_err, energy_err = [], []
for n in ns:
    p = u.state(t_star, n_trotter=int(n)); p /= np.linalg.norm(p)
    if float(psi @ p) < 0:
        p = -p
    state_err.append(np.linalg.norm(p - psi))
    energy_err.append(u.energy(t_star, n_trotter=int(n)) - ref["energy"])

fig, ax = plt.subplots(figsize=(7, 4.4))
ax.loglog(ns, state_err, "o-", label=r"$\|\Psi_n - \Psi\|$")
ax.loglog(ns, energy_err, "s-", label="energy error")
ax.loglog(ns, state_err[0] / ns, "k--", lw=0.8, label=r"$\propto 1/n$")
ax.loglog(ns, energy_err[0] / ns**2, "k:", lw=0.8, label=r"$\propto 1/n^2$")
ax.set_xlabel("Trotter steps $n$"); ax.set_ylabel("error")
ax.set_title("amplitudes frozen at their exact-exponential values")
ax.legend(); fig.tight_layout(); plt.show()

Two lessons.  **Re-optimising** the amplitudes at each Trotter number gives the
same energy to ten digits, even at $n=1$ — the optimiser absorbs the splitting
error into the amplitudes.  **Freezing** them exposes the error, and it falls
as $1/n$ in the state but $1/n^2$ in the energy, because the amplitudes sit at
a variational stationary point so the first-order response vanishes.

Both say the same thing: a variational algorithm is far more forgiving of
Trotter error than a simulation algorithm.  That is a practical argument for
VQE over straight time evolution on near-term hardware, where circuit depth is
the binding constraint.

## 9. UCCSD, where the singles matter

The pairing plus particle-hole model of Chapter 7 again.  Here the singles
generators do real work, and the contrast between the variational and the
standard theory is sharpest.

In [ ]:
cc.demo_uccsd()

### This is a VQE

What we have just run *is* a variational quantum eigensolver, on a classical
simulator: a parametrised trial state $e^{\hat\sigma(\mathbf t)}|\Phi_0\rangle$
Trotterised into a gate sequence, an energy evaluated on it, and a classical
optimiser adjusting $\mathbf t$.  On hardware the energy evaluation is replaced
by repeated measurement in bases fixed by the Pauli decomposition of the
Hamiltonian.  Nothing else changes — and the `UnitaryCC` class used here is the
one Chapter 11 will build on.

## The full program

Everything above lives in
`BookManybody/BookMaterial/Programs/coupledcluster.py`, which runs as a
script and prints all six demonstrations of the chapter.

In [ ]:
print(open(cc.__file__).read())